# CineMatch-AI — Data prep

Explore raw MovieLens data, load cached TMDb metadata, then build **train / val / test** (70 / 10 / 20 temporal split) under `data/processed/`.

Only those three parquet files are committed to GitHub (via Git LFS). Intermediate joins stay in memory.

Run cells top-to-bottom. TMDb fetching stays in the terminal (`scripts/fetch_tmdb_metadata.py`). Paths work from the repo root or `Notebooks/`.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data").is_dir() and (ROOT.parent / "data").is_dir():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

c:\Users\ahmed\Anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
ratings_df = pd.read_csv(DATA_DIR / "ratings.csv")

In [ ]:
print(ratings_df.shape)
ratings_df.head()

In [ ]:
ratings_df.isna().sum()

In [ ]:
ratings_df['userId'].nunique()

In [ ]:
movies_df = pd.read_csv(DATA_DIR / "movies.csv")
print(movies_df.shape)
movies_df.head()

In [ ]:
movies_df.isna().sum()

In [ ]:
print(movies_df['movieId'].nunique())
print(ratings_df['movieId'].nunique())

In [ ]:
link_df = pd.read_csv('data/links.csv')
print(link_df.shape)
link_df.head()

In [ ]:
link_df.isna().sum()

In [ ]:
print(link_df['movieId'].nunique())
print(ratings_df['movieId'].nunique())

In [ ]:
tag_df = pd.read_csv(DATA_DIR / "tags.csv")

In [ ]:
print(tag_df.shape)
tag_df.head()

## TMDb metadata (load from cache)

Fetching is handled by `scripts/fetch_tmdb_metadata.py` — don't run fetch cells here.

**Fetch / resume in terminal:**
```powershell
python scripts/fetch_tmdb_metadata.py --only-rated
python scripts/fetch_tmdb_metadata.py --merge-only
```

Then run the cells below to load the saved parquet files.

In [ ]:
from pathlib import Path

metadata_path = Path("data/metadata_df.parquet")
if not metadata_path.exists():
    raise FileNotFoundError(
        "No metadata cache yet. Run: python scripts/fetch_tmdb_metadata.py --only-rated"
    )

metadata_df = pd.read_parquet(metadata_path)
print(metadata_df.shape)
metadata_df.head(3)

In [ ]:
metadata_df.isna().sum()

## Build train / val / test

Split ratings by time, join metadata one split at a time, write `train.parquet`, `val.parquet`, and `test.parquet`. Overview is omitted (too large at ~32M rows).

In [ ]:
import sys

sys.path.insert(0, str(ROOT))

import importlib
import scripts.data_helpers as data_helpers
importlib.reload(data_helpers)

build_movies_catalog = data_helpers.build_movies_catalog
build_and_save_splits = data_helpers.build_and_save_splits

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2
BATCH_SIZE = 1_000_000  # rows per parquet write batch

In [ ]:
catalog = build_movies_catalog(DATA_DIR / "movies.csv", DATA_DIR)
print(f"Movies catalog (in memory): {len(catalog):,} movies  |  {catalog.shape[1]} columns")
catalog.head(3)

In [ ]:
print("Splitting ratings and writing train / val / test (one split at a time)...")
result = build_and_save_splits(
    DATA_DIR / "ratings.csv",
    catalog,
    PROCESSED_DIR,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    slim=True,  # omit overview — too large to duplicate on ~32M rows
    batch_size=BATCH_SIZE,
)

print(f"Val cutoff timestamp:  {result['val_cutoff']:.0f}")
print(f"Test cutoff timestamp: {result['test_cutoff']:.0f}")

## Verify train / val / test

Reload from disk to confirm the three committed parquet files.

In [ ]:
train_df = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val_df = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test_df = pd.read_parquet(PROCESSED_DIR / "test.parquet")

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)
train_df.head()

In [ ]:
print(metadata_df["fetch_status"].value_counts())
print(f"Successful fetches: {(metadata_df['fetch_status'] == 'ok').sum():,} / {len(metadata_df):,}")